# Week 1 · Day 2 — One API shape, many providers

The OpenAI Chat Completions format has become a de-facto standard, so the
*same* request shape can talk to very different backends. This notebook makes
the identical call four ways:

1. **Raw HTTP** — what the SDK does under the hood.
2. **The official OpenAI SDK** — the convenient wrapper.
3. **Google Gemini** — via its OpenAI-compatible endpoint.
4. **A local model with Ollama** — same shape, running on your machine.

The only things that change between providers are the **base URL**, **API key**,
and **model name**.

> Needs `OPENAI_API_KEY` (and `GEMINI_API_KEY` for the Gemini cell). The Ollama
> cells require a local [Ollama](https://ollama.com) server running.

In [13]:
import os

import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()
key = os.getenv("OPENAI_API_KEY")
if not key:
    raise ValueError("OPENAI_API_KEY not found in environment variables.")

## 1. Calling the API directly over HTTP

Before reaching for the SDK, it's worth seeing the bare request. The
Chat Completions endpoint is a plain `POST` with a Bearer token and a JSON
body — nothing more. Authentication and the payload shape here are exactly
what the SDK builds for you in the next section.

In [3]:
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {key}",
}

payload = {
    "model": "gpt-5-nano",
    "messages": [{"role": "user", "content": "Tell me a fun fact."}],
}

In [ ]:
# The endpoint returns plain JSON, so we navigate the response dict by hand to
# pull out the generated text. The SDK (next cell) does this unwrapping for us.
response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload).json()
print(response["choices"][0]["message"]["content"])

## 2. Using the OpenAI Python SDK

The SDK wraps that same HTTP call: it builds the headers, sends the request,
and hands back a typed object so we can write `response.choices[0].message.content`
instead of indexing a raw dict.

In [15]:
from openai import OpenAI

client = OpenAI(api_key=key)
response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[{"role": "user", "content": "Tell me a fun fact about India."}],
)

display(Markdown(response.choices[0].message.content))

Fun fact: The Iron Pillar of Delhi, built around 400–450 CE, has stayed rust-free for about 1,600 years in a humid climate—an amazing example of ancient Indian metallurgy. Want another fun fact about India?

## 3. Same SDK, a different provider — Google Gemini

Here's the payoff of a shared API shape: point the OpenAI client at Gemini's
OpenAI-compatible base URL, swap the key and model, and **everything else is
identical**. No new SDK to learn.

In [ ]:
# Only base_url, api_key, and model change — the call below is byte-for-byte
# the same as the OpenAI one, because Gemini speaks the OpenAI wire format.
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key=os.getenv("GEMINI_API_KEY"))
response = gemini.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=[{"role": "user", "content": "Tell me a fun fact about India."}],
)

display(Markdown(response.choices[0].message.content))

## 4. Running locally with Ollama

Same trick, but the backend now runs on your own machine. We first pull a
model, then point the client at Ollama's local OpenAI-compatible server. Great
for offline work, privacy, and zero per-token cost.

In [ ]:
!ollama pull llama3.2:1b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏  48 KB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 598 KB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 664 KB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕         

In [ ]:
# Ollama requires an api_key argument but ignores its value, so any non-empty
# placeholder works. The local server speaks the same OpenAI Chat Completions API.
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
response = ollama.chat.completions.create(
    model="llama3.2:3b",
    messages=[{"role": "user", "content": "Tell me a fun fact about India."}],
)

display(Markdown(response.choices[0].message.content))